# 01_explore_ookla Explore the Ookla data

**Owner:** Person 1

**Goal:** Understand the Hanoi speed data before building anything.

**Outputs:** figures in artifacts/reports/, findings written at the end of this notebook

In [ ]:
import sys
from pathlib import Path

# Cho phép import package speedmap khi chạy notebook từ thư mục notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from speedmap import config

## 1. Load

In [ ]:
from speedmap.data import ookla

raw = ookla.load_raw()
config.ARTIFACTS_REPORTS.mkdir(parents=True, exist_ok=True)
print(len(raw), "rows,", raw.quadkey.nunique(), "tiles")
raw.head()

## 2. Overview per quarter

In [ ]:
ookla.summary(raw)

## 3. Distributions
Download, upload, latency, tests per tile. Note how many tiles have only 1 test.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["download_mbps", "upload_mbps", "latency_ms", "tests"]):
    ax.hist(raw[col].clip(lower=0.1), bins=np.logspace(-1, np.log10(raw[col].max() + 1), 50))
    ax.set_xscale("log"); ax.set_title(col)
plt.tight_layout(); plt.savefig(config.ARTIFACTS_REPORTS / "eda_distributions.png", dpi=150); plt.show()

print(f"rows with 1 test: {(raw.tests == 1).mean():.1%}, with at least 3 tests: {(raw.tests >= 3).mean():.1%}")
quarters_per_tile = raw.groupby("quadkey").quarter.nunique()
print(f"tiles seen in all 6 quarters: {(quarters_per_tile == 6).mean():.1%}, in only 1 quarter: {(quarters_per_tile == 1).mean():.1%}")

## 4. Map
Scatter of tile centres coloured by download speed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(raw.longitude, raw.latitude, c=raw.download_mbps.clip(upper=300), s=3, cmap="viridis")
plt.colorbar(sc, label="download (Mbps, clipped at 300)")
ax.set_title("Ookla mobile download speed, Hanoi"); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.savefig(config.ARTIFACTS_REPORTS / "eda_map.png", dpi=150); plt.show()

# Tốc độ theo khoảng cách tới trung tâm (Hoàn Kiếm)
lat0, lon0 = config.HANOI_CENTER
dist_km = np.hypot((raw.latitude - lat0) * 110.57, (raw.longitude - lon0) * 111.32 * np.cos(np.radians(lat0)))
bands = pd.cut(dist_km, [0, 5, 10, 20, 100], labels=["0-5 km", "5-10 km", "10-20 km", ">20 km"])
raw.groupby(bands, observed=True).download_mbps.agg(["count", "median"]).round(1)

## 5. Findings
- **Data size:** 30,214 tile-quarter rows covering 10,459 tiles over six quarters (Q4 2024 to Q1 2026); after cleaning 30,076 rows remain.
- **Noisy tiles:** 37.6% of rows contain only one speed test and only 46.4% have at least 3 tests, so we weight rows by √tests and evaluate on tiles with at least 3 tests.
- **Strong spatial pattern:** median download is about 146 Mbps within 5 km of Hoan Kiem, 112 Mbps at 5–10 km, 69 Mbps at 10–20 km and 62 Mbps beyond 20 km. Neighbouring tiles are similar, so we must use spatial block cross-validation.
- **Speeds rise over time:** median download went from 70.6 Mbps (Q4 2024) to 79.2 Mbps (Q1 2026), which supports quarterly retraining.
- **Tiles are not stable over time:** only 16.3% of tiles appear in all six quarters and 34.1% appear in only one, so the model must generalise to tiles it has not seen.